In [7]:
# %% [markdown]
# # 모두마켓 분기 성과 요약 보고서
# 단계: 데이터 준비 → 키 검증 → 병합 → 집계 → 특성 추가 → 보고서

# %%
import pandas as pd
import numpy as np

# %% [markdown]
# ## 1. 데이터 준비
# 종합 실습 데이터 생성 셀 (그대로 사용)

# %%
np.random.seed(7)
n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})

# 이후 단계에서 쓰기 편하게 이름 정리
customers = cust.copy()
products = prod.copy()
orders = ordr.copy()

print("스냅샷 준비 완료 — orders:", orders.shape, "| customers:", customers.shape, "| products:", products.shape)

# %% [markdown]
# ## 2. 키 검증 (필수)
# 병합 전에 반드시 확인해야 하는 세 가지:
# 1) **중복 키** — 병합 기준 테이블(customers, products)에서 키가 유일한가?
# 2) **미매칭 키** — orders에 있는 키가 상대 테이블에 전부 존재하는가?
# 3) **관계(cardinality)** — 실제로 m:1(여러 주문 : 한 고객/상품) 구조가 맞는가?

# %%
# 검증 1) 중복 키
dup_cust = customers["customer_id"].duplicated().sum()
dup_prod = products["product_id"].duplicated().sum()
print("customers.customer_id 중복 개수:", dup_cust)
print("products.product_id 중복 개수:", dup_prod)

# %%
# 검증 2) 미매칭 키 (orders 쪽 키가 상대 테이블에 없는 경우)
unmatched_cust = set(orders["customer_id"]) - set(customers["customer_id"])
unmatched_prod = set(orders["product_id"]) - set(products["product_id"])
print("orders에만 있고 customers에 없는 customer_id 개수:", len(unmatched_cust))
print("orders에만 있고 products에 없는 product_id 개수:", len(unmatched_prod))

# %%
# 검증 3) 관계 확인 — orders는 여러 건, customers/products는 유일해야 m:1
print("orders 전체 건수:", len(orders))
print("orders.customer_id 유니크 수:", orders["customer_id"].nunique(),
      " / customers 유니크 수:", customers["customer_id"].nunique())
print("orders.product_id 유니크 수:", orders["product_id"].nunique(),
      " / products 유니크 수:", products["product_id"].nunique())

# %% [markdown]
# **검증 결과:** customers·products 양쪽 모두 키 중복 0건, orders의 customer_id·product_id
# 미매칭 0건이므로 **m:1 병합이 안전하다.** (orders = 다수, customers/products = 유일 키)

# %% [markdown]
# ## 3. 병합
# `validate="m:1"`을 걸어 orders(다수) : customers/products(유일) 관계를 강제 확인

# %%
merged = (
    orders
    .merge(customers, on="customer_id", how="left", validate="m:1")
    .merge(products, on="product_id", how="left", validate="m:1")
)
print("병합 후 shape:", merged.shape)
print("\n결측치 확인:\n", merged.isna().sum())
merged.head()

# %% [markdown]
# ## 4. 집계

# %% [markdown]
# ### 4-1. 지역 × 회원등급 매출 교차표 (pivot_table)

# %%
region_membership_pivot = pd.pivot_table(
    merged,
    index="region",
    columns="membership",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="합계",
)
region_membership_pivot

# %% [markdown]
# ### 4-2. 카테고리별 KPI 요약표 (groupby + agg)
# 매출 · 건수 · 객단가(평균 주문금액)

# %%
category_kpi = (
    merged
    .groupby("category")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
    )
    .sort_values("총매출", ascending=False)
)
category_kpi["객단가"] = category_kpi["객단가"].round(0)
category_kpi


스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)
customers.customer_id 중복 개수: 0
products.product_id 중복 개수: 0
orders에만 있고 customers에 없는 customer_id 개수: 0
orders에만 있고 products에 없는 product_id 개수: 0
orders 전체 건수: 1500
orders.customer_id 유니크 수: 200  / customers 유니크 수: 200
orders.product_id 유니크 수: 30  / products 유니크 수: 30
병합 후 shape: (1500, 10)

결측치 확인:
 order_id       0
customer_id    0
product_id     0
quantity       0
amount         0
order_date     0
region         0
membership     0
category       0
price          0
dtype: int64


,총매출,주문건수,객단가
category,,,
패션,38318000.0,512,74840.0
식품,22404000.0,398,56291.0
가전,20248000.0,398,50874.0
뷰티,9986000.0,192,52010.0


In [6]:
python -c "import sys; print(sys.executable)"
python -c "import pandas; print(pandas.__version__)"

SyntaxError: invalid syntax (4278255368.py, line 1)